# Inspect SFT `input_text` and `output_text`

Temporary debugging notebook. It reproduces the relevant part of `CustomTrainer.compute_loss`: decode `inputs["input_ids"]`, run one forward pass, argmax-decode `outputs.logits`, and print both lists.


In [2]:
# Pick the physical GPU(s) before importing torch.
# If you already imported torch in this kernel, restart the kernel and run from here.
import os
CUDA_VISIBLE_DEVICES = '2'  # edit to '3', '4', '5', or '2,3,4,5' if needed
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from pathlib import Path
import sys
import torch

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from utils.dataset_utils import prepare_train_data, format_dataset_text

print(REPO_ROOT)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('cuda visible:', torch.cuda.device_count(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


/mnt/disk2/gzt/RL4DistReconfig
CUDA_VISIBLE_DEVICES: 2
cuda visible: 1 NVIDIA H800


In [3]:
# Edit these before running the model cell. Keep these small for inspection.
DATA_PATH = 'Dataset/Processed/train_33_69_84_nodes.csv'
MODEL_ID = '../models/meta-llama/Llama-3.1-8B-Instruct'
PROMPT_FORMAT = 'llama3_chat'  # legacy, qwen_chat, llama3_chat
SPLIT = 'train'
BATCH_SIZE = 2
MAX_SEQ_LENGTH = 512
COMPUTE_LOSS_AND_GRAD = True
ATTACH_LORA_FOR_GRAD_CHECK = True


In [4]:
train_ds, val_ds, test_ds = prepare_train_data(DATA_PATH)
split_map = {'train': train_ds, 'validation': val_ds, 'test': test_ds}
dataset = format_dataset_text(split_map[SPLIT], PROMPT_FORMAT)

print(dataset)
print(dataset.column_names)
print(dataset[0]['text'][:1200])


/mnt/disk2/gzt/envs/grpo_gzt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 17520/17520 [00:01<00:00, 15454.96 examples/s]

Dataset({
    features: ['id', 'Task Description', 'input', 'prompt', 'output', 'text'],
    num_rows: 17520
})
['id', 'Task Description', 'input', 'prompt', 'output', 'text']
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

 
Find the optimal configuration, i.e. the optimal connectivity and optimal open lines of these buses and lines 
so as to ensure energy distribution to the whole system while minimizing the power loss. The number given for the busses indicates the 
total number of busses starting from 1 going all the way to the given number in increments of 1. Make sure the Open Lines 
in the output include ONLY Lines that are given in the input and that you take into account their given properties. 
The Available Lines WITHOUT the Open Lines should form a network graph that is a single graph, i.e. no subgraphs or 
multiple connected components lists and the graph should NOT contain any cycles i.e. the number of available lines WITHOUT 
the number of open lines should EQU

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.use_cache = False

if COMPUTE_LOSS_AND_GRAD and ATTACH_LORA_FOR_GRAD_CHECK:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.0,
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

model.eval()
print(type(model))


Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.89s/it]


trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424
<class 'peft.peft_model.PeftModelForCausalLM'>


In [15]:
batch.keys()

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [6]:
texts = [dataset[i]['text'] for i in range(BATCH_SIZE)]
batch = tokenizer(
    texts,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
)
batch = {k: v.to(model.device) for k, v in batch.items()}

if COMPUTE_LOSS_AND_GRAD:
    batch['labels'] = batch['input_ids'].clone()
    outputs = model(**batch)
else:
    # Do not pass labels when you only want decoded logits; CE loss can use much more memory.
    with torch.inference_mode():
        outputs = model(**batch)

input_text = tokenizer.batch_decode(batch['input_ids'], skip_special_tokens=True)
output_text = tokenizer.batch_decode(outputs.logits.argmax(dim=-1), skip_special_tokens=True)

print('has loss:', outputs.loss is not None)
if outputs.loss is not None:
    print('ce loss:', float(outputs.loss.detach().float().cpu()))
print('input_text type:', type(input_text), 'len:', len(input_text))
print('output_text type:', type(output_text), 'len:', len(output_text))
print('logits shape:', tuple(outputs.logits.shape))


has loss: True
ce loss: 2.300776720046997
input_text type: <class 'list'> len: 2
output_text type: <class 'list'> len: 2
logits shape: (2, 512, 128256)


In [12]:
texts

['<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n \nFind the optimal configuration, i.e. the optimal connectivity and optimal open lines of these buses and lines \nso as to ensure energy distribution to the whole system while minimizing the power loss. The number given for the busses indicates the \ntotal number of busses starting from 1 going all the way to the given number in increments of 1. Make sure the Open Lines \nin the output include ONLY Lines that are given in the input and that you take into account their given properties. \nThe Available Lines WITHOUT the Open Lines should form a network graph that is a single graph, i.e. no subgraphs or \nmultiple connected components lists and the graph should NOT contain any cycles i.e. the number of available lines WITHOUT \nthe number of open lines should EQUAL the number of busses minus one. If you predict the system loss and the value is greater \nthan the current system loss, DO NOT reconfigure the network and return 

In [21]:
input_text[1]

'user\n\n \nFind the optimal configuration, i.e. the optimal connectivity and optimal open lines of these buses and lines \nso as to ensure energy distribution to the whole system while minimizing the power loss. The number given for the busses indicates the \ntotal number of busses starting from 1 going all the way to the given number in increments of 1. Make sure the Open Lines \nin the output include ONLY Lines that are given in the input and that you take into account their given properties. \nThe Available Lines WITHOUT the Open Lines should form a network graph that is a single graph, i.e. no subgraphs or \nmultiple connected components lists and the graph should NOT contain any cycles i.e. the number of available lines WITHOUT \nthe number of open lines should EQUAL the number of busses minus one. If you predict the system loss and the value is greater \nthan the current system loss, DO NOT reconfigure the network and return the same configuration as in the input. ONLY \nreturn 

In [7]:
for i, (inp, out) in enumerate(zip(input_text, output_text)):
    print('=' * 100)
    print(f'SAMPLE {i} input_text, first 2000 chars')
    print(inp[:2000])
    print('-' * 100)
    print(f'SAMPLE {i} output_text, first 2000 chars')
    print(out[:2000])


SAMPLE 0 input_text, first 2000 chars
user

 
Find the optimal configuration, i.e. the optimal connectivity and optimal open lines of these buses and lines 
so as to ensure energy distribution to the whole system while minimizing the power loss. The number given for the busses indicates the 
total number of busses starting from 1 going all the way to the given number in increments of 1. Make sure the Open Lines 
in the output include ONLY Lines that are given in the input and that you take into account their given properties. 
The Available Lines WITHOUT the Open Lines should form a network graph that is a single graph, i.e. no subgraphs or 
multiple connected components lists and the graph should NOT contain any cycles i.e. the number of available lines WITHOUT 
the number of open lines should EQUAL the number of busses minus one. If you predict the system loss and the value is greater 
than the current system loss, DO NOT reconfigure the network and return the same configuration as i

In [8]:
from utils.metrics_utils import parse_available_lines, parse_open_lines

for i, (inp, out) in enumerate(zip(input_text, output_text)):
    print('=' * 100)
    print('sample', i)
    print('available_lines[:5]:', parse_available_lines(inp)[:5])
    print('parsed open lines:', parse_open_lines(out))


sample 0
available_lines[:5]: []
parsed open lines: []
sample 1
available_lines[:5]: []
parsed open lines: []


In [9]:
# Exact reproduction of the relevant CustomTrainer.compute_loss lines.
# The important part is that input_text/output_text are lists, but the
# author's custom graph loss reads only input_text[0] and output_text[0].
from SFT.reproduce_author_llama import filter_predicted_lines
from utils.metrics_utils import (
    parse_available_lines, parse_open_lines, get_output_graph_edges,
)

inputs = batch
outputs = model(**inputs)
loss = outputs.loss

input_text = tokenizer.batch_decode(
    inputs['input_ids'], skip_special_tokens=True
)
output_text = tokenizer.batch_decode(
    outputs.logits.argmax(dim=-1), skip_special_tokens=True
)

available_lines = parse_available_lines(input_text[0])
predicted_lines = parse_open_lines(output_text[0])
predicted_lines = filter_predicted_lines(predicted_lines)
graph_edges = get_output_graph_edges(predicted_lines, available_lines)

print('loss:', float(loss.detach().float().cpu()))
print('type(input_text):', type(input_text), 'len:', len(input_text))
print('type(output_text):', type(output_text), 'len:', len(output_text))
print('CUSTOM LOSS USES ONLY SAMPLE INDEX: 0')
print('available_lines from input_text[0], count:', len(available_lines))
print('predicted_lines from output_text[0]:', predicted_lines)
print('graph_edges count:', len(graph_edges))

for i, (inp, out) in enumerate(zip(input_text, output_text)):
    sample_available = parse_available_lines(inp)
    sample_predicted = filter_predicted_lines(parse_open_lines(out))
    print('=' * 100)
    print(f'sample {i}')
    print('available count:', len(sample_available))
    print('predicted:', sample_predicted)
    print('input_text prefix:', inp[:300].replace('\n', ' '))
    print('output_text prefix:', out[:300].replace('\n', ' '))


loss: 2.300776720046997
type(input_text): <class 'list'> len: 2
type(output_text): <class 'list'> len: 2
CUSTOM LOSS USES ONLY SAMPLE INDEX: 0
available_lines from input_text[0], count: 0
predicted_lines from output_text[0]: []
graph_edges count: 0
sample 0
available count: 0
predicted: []
input_text prefix: user    Find the optimal configuration, i.e. the optimal connectivity and optimal open lines of these buses and lines  so as to ensure energy distribution to the whole system while minimizing the power loss. The number given for the busses indicates the  total number of busses starting from 1 going 
output_text prefix: QuestionQuestion  ### the value solution of i.e., the configuration values and the node-loop, the lines, trains,that that to minimize that efficiency to all load area. minimizing the energy loss. optimal of in each busesusses is the powernumber power of linesusses in from the1 to to the way to  last
sample 1
available count: 0
predicted: []
input_text prefix: user   

In [10]:
# Gradient diagnostic: compare CE gradient with CE + discrete custom graph penalty.
# This checks all trainable LoRA parameters, not just the first grad tensor.
from utils.metrics_utils import (
    parse_available_lines, parse_open_lines, get_output_graph_edges,
    compute_invalid_edges_loss, compute_cycles_loss, compute_subgraphs_loss,
)

def trainable_grad_report(model, top_k=8):
    total_sq = 0.0
    rows = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.grad is None:
            rows.append((name, None))
            continue
        norm = p.grad.detach().float().norm().item()
        rows.append((name, norm))
        total_sq += norm * norm
    non_none = [(name, norm) for name, norm in rows if norm is not None]
    nonzero = [(name, norm) for name, norm in non_none if norm > 0]
    top = sorted(non_none, key=lambda item: item[1], reverse=True)[:top_k]
    return {
        'total_norm': total_sq ** 0.5,
        'trainable_count': len(rows),
        'grad_count': len(non_none),
        'nonzero_count': len(nonzero),
        'top': top,
    }

if not COMPUTE_LOSS_AND_GRAD:
    print('Set COMPUTE_LOSS_AND_GRAD=True and rerun the forward cell first.')
else:
    sample_penalties = []
    for inp, out in zip(input_text, output_text):
        available_lines = parse_available_lines(inp)
        predicted_lines = parse_open_lines(out)
        graph_edges = get_output_graph_edges(predicted_lines, available_lines)
        if predicted_lines and available_lines:
            invalid = compute_invalid_edges_loss(predicted_lines, available_lines) / len(predicted_lines)
            cycles = compute_cycles_loss(graph_edges) / len(available_lines)
            subgraphs = compute_subgraphs_loss(graph_edges) / len(predicted_lines)
        else:
            invalid = cycles = subgraphs = 1.0
        sample_penalties.append(float(invalid) + float(cycles) + float(subgraphs))
    custom_penalty = sum(sample_penalties) / max(len(sample_penalties), 1)
    total_loss = outputs.loss + custom_penalty

    print('custom_penalty:', custom_penalty, type(custom_penalty))
    print('ce loss:', float(outputs.loss.detach().float().cpu()))
    print('ce requires_grad:', outputs.loss.requires_grad)
    print('total requires_grad:', total_loss.requires_grad)

    model.zero_grad(set_to_none=True)
    outputs.loss.backward(retain_graph=True)
    ce_report = trainable_grad_report(model)

    model.zero_grad(set_to_none=True)
    total_loss.backward()
    total_report = trainable_grad_report(model)

    print('CE grad report:', ce_report)
    print('CE + custom grad report:', total_report)
    print('total_norm difference:', total_report['total_norm'] - ce_report['total_norm'])
    print('top CE grads:')
    for name, norm in ce_report['top']:
        print(f'  {norm:.8g}  {name}')
    print('top CE + custom grads:')
    for name, norm in total_report['top']:
        print(f'  {norm:.8g}  {name}')


custom_penalty: 3.0 <class 'float'>
ce loss: 2.300776720046997
ce requires_grad: True
total requires_grad: True
CE grad report: {'total_norm': 0.8466149814617798, 'trainable_count': 128, 'grad_count': 128, 'nonzero_count': 64, 'top': [('base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 0.39424842596054077), ('base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight', 0.38616347312927246), ('base_model.model.model.layers.3.self_attn.v_proj.lora_B.default.weight', 0.26679372787475586), ('base_model.model.model.layers.2.self_attn.v_proj.lora_B.default.weight', 0.23633268475532532), ('base_model.model.model.layers.4.self_attn.v_proj.lora_B.default.weight', 0.1780823916196823), ('base_model.model.model.layers.6.self_attn.v_proj.lora_B.default.weight', 0.1543356329202652), ('base_model.model.model.layers.7.self_attn.v_proj.lora_B.default.weight', 0.15049560368061066), ('base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight', 0.13819789886